In [ ]:
%pip install numpy matplotlib nbconvert scipy > /dev/null
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import kalman
kalman.set_debug(0)

import kalman.runtime.chrono as chrono
import numpy as np
import matplotlib.pyplot as plt
from kalman.helpers import rainbow

# Sensor Fusion with a Kalman Filter

One latent target, several **asynchronous, differently-noisy** sensors. The Kalman update weights
each sensor by its inverse covariance, so the fused estimate beats **any single sensor** — and
different rates fall out for free: a sensor that does not fire on a step is a `NaN` row, which the
engine simply skips (predict-only for that channel).

`kalman.apps.fusion.SensorFusionModel` tracks a constant-velocity target $x=[\,p,\ v\,]$ observed by
a configurable list of `SensorSpec` (each declares what it observes, its noise and its rate). The
same model covers two regimes: **redundant** position sensors (the stereo-camera + lidar analog) and
**complementary** position + velocity (the GPS + IMU analog).

## 1 — Redundant position sensors

Three position sensors of comparable but different quality and rate: a noisy slow **GPS** (3 m,
20 Hz), a precise fast **radar** (2 m, 50 Hz) and a medium **lidar** (2.5 m, 25 Hz). Their rows stack
into $H$ and their variances into $R=\mathrm{diag}(\sigma_i^2)$.

In [2]:
from kalman.apps.fusion import SensorFusionModel, SensorSpec

fs = 50.0
sensors = [
    SensorSpec("GPS",   "position", uncertainty=3.0, rate=20),
    SensorSpec("Radar", "position", uncertainty=2.0, rate=50),
    SensorSpec("Lidar", "position", uncertainty=2.5, rate=25),
]
model = SensorFusionModel("target", sensors, fs=fs, process_uncertainty=2.0)
model.initial_state([0.0, 8.0])          # start at x=0 moving 8 m/s
print(f"state dim n={model.n}  measurement dim m={model.m}  (one row per sensor)")

state dim n=2  measurement dim m=3  (one row per sensor)


## 2 — Build the asynchronous stream and fuse

`make_measurements` samples each sensor at its own rate, adds its own noise, and interleaves the
streams into one `NaN`-padded matrix at the base rate. The signature figure shows the per-sensor
scatter, the fused estimate ($\pm\sigma$), and — crucially — the fused position $\sigma$ against
every single-sensor $\sigma$.

In [ ]:
truth = model.sample(8.0)                       # clean constant-velocity ground truth
zk = model.make_measurements(truth, seed=1)     # multi-rate, NaN-padded
print("measurement matrix:", zk.shape, " NaN fraction per sensor:",
      np.round(np.isnan(zk).mean(axis=0), 2))

record = model.filter(zk)
model.plot(record, sample=truth, measurements=zk,
           meas=dict(alpha=1.0), colors=rainbow(len(model.sensors), l=0.6))
plt.show()

## 3 — Fusion beats every single sensor

Filtering each sensor alone (all other channels masked to `NaN`) gives the baseline each could
achieve on its own. The fused estimate has both smaller steady-state variance and smaller RMSE.

In [4]:
tr_pos = truth.xk[:, 0]
def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

print(f"{'sensor':<10}{'final sigma':>12}{'pos RMSE':>12}")
for i, s in enumerate(sensors):
    ri = model.single_sensor_record(zk, i)
    print(f"{s.name:<10}{np.sqrt(ri.Pk[-1,0,0]):>12.3f}{rmse(ri.xk[:,0], tr_pos):>12.3f}")
print(f"{'FUSED':<10}{np.sqrt(record.Pk[-1,0,0]):>12.3f}{rmse(record.xk[:,0], tr_pos):>12.3f}")

sensor     final sigma    pos RMSE
GPS              1.054       0.760
Radar            0.538       0.552
Lidar            0.841       0.851
FUSED            0.465       0.458


## 4 — Complementary fusion: position + velocity

The classic complementary filter: a slow noisy **GPS** position fused with a fast **IMU/encoder**
velocity. Neither alone pins the position well — GPS is noisy, integrating velocity drifts — but
together they are tight.

In [ ]:
comp = SensorFusionModel("car", [
    SensorSpec("GPS", "position", uncertainty=6.0, rate=5),    # slow, noisy position
    SensorSpec("IMU", "velocity", uncertainty=0.3, rate=50),   # fast, precise velocity
], fs=50.0, process_uncertainty=1.0)
comp.initial_state([0.0, 10.0])

truth2 = comp.sample(8.0)
z2 = comp.make_measurements(truth2, seed=3)
rec2 = comp.filter(z2)
print(f"GPS-only final pos sigma = {np.sqrt(comp.single_sensor_record(z2,0).Pk[-1,0,0]):.3f} m")
print(f"fused      final pos sigma = {np.sqrt(rec2.Pk[-1,0,0]):.3f} m")
comp.plot(rec2, sample=truth2, measurements=z2,
          meas=dict(alpha=1.0), colors=rainbow(len(comp.sensors), l=0.6))
plt.show()

## 5 — Why it works: inverse-variance weighting

For two **synchronous** position readings the optimal single-step fusion is
$\hat x = (\sigma_2^2 z_1 + \sigma_1^2 z_2)/(\sigma_1^2+\sigma_2^2)$ with variance
$1/(1/\sigma_1^2+1/\sigma_2^2)$ — always smaller than either input. The Kalman update is exactly this,
generalised over time and across asynchronous sensors.

In [6]:
s1, s2 = 3.0, 4.0
fused_var = 1.0 / (1.0/s1**2 + 1.0/s2**2)
print(f"sigma1={s1}  sigma2={s2}  ->  fused sigma = {fused_var**0.5:.3f}  (< {min(s1,s2)})")
print("inverse-variance weights:", np.round([ (1/s1**2)/(1/s1**2+1/s2**2),
                                               (1/s2**2)/(1/s1**2+1/s2**2) ], 3))

sigma1=3.0  sigma2=4.0  ->  fused sigma = 2.400  (< 3.0)
inverse-variance weights: [0.64 0.36]


## Takeaways

- Sensor fusion is just a Kalman **measurement update** with a stacked $H$ / block-diagonal $R$.
- Asynchronous, multi-rate sensors need no special machinery: missing rows are `NaN` (predict-only).
- The fused estimate beats every single sensor — redundant sensors shrink the variance, complementary
  sensors observe different states (position + velocity) and together pin the full state.